# 🚚 HotWAY CVRP Production Benchmark & Statistical Analysis

> **Dự án:** HotWAY v5.0 — Hệ thống giải bài toán CVRP giao đồ ăn nóng trên bản đồ TP.HCM thực tế
> **Vai trò:** AI Data Engineer
> **Dữ liệu:** Đồ thị OpenStreetMap TP.HCM (46,055 nút, 107,224 cạnh, 8,551 km đường)
> **Mục tiêu:** Tự tay sinh dữ liệu, huấn luyện DQN/Double DQN, chạy benchmark 30 instances, và kiểm định thống kê


## 1. Thiết Lập Môi Trường & Thư Viện

In [ ]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import torch

# Đổi thư mục làm việc về workspace root nếu cần
if not os.path.exists('config/site.yaml'):
    os.chdir('..')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 2. Phân Tích Cấu Trúc Đồ Thị Đường Bộ TP.HCM (OSMnx Directed Graph)
Kiểm tra tính liên thông mạnh (SCC) và phân bố chiều dài các cạnh đường bộ.

In [ ]:
graph_path = 'data/hcmc_compact_graph.pickle'
if os.path.exists(graph_path):
    with open(graph_path, 'rb') as f:
        G = pickle.load(f)
    print(f"[OK] Đồ thị TP.HCM loaded:")
    print(f"  - Số nút: {G.number_of_nodes():,}")
    print(f"  - Số cạnh có hướng: {G.number_of_edges():,}")
    print(f"  - Liên thông mạnh (SCC): {nx.is_strongly_connected(G)}")
    
    lengths = [d.get('length', 0) for _, _, d in G.edges(data=True)]
    print(f"  - Tổng chiều dài đường bộ: {sum(lengths)/1000:,.1f} km")
    print(f"  - Chiều dài trung bình cạnh: {np.mean(lengths):.1f} m")
    
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].hist(lengths, bins=50, range=(0, 500), color='#1e88e5', edgecolor='white')
    ax[0].set_title('Phân Bố Chiều Dài Cạnh Đường Bộ (0 - 500m)')
    ax[0].set_xlabel('Chiều dài (m)')
    ax[0].set_ylabel('Số lượng cạnh')
    
    degrees = [d for _, d in G.out_degree()]
    ax[1].hist(degrees, bins=range(1, 8), color='#43a047', edgecolor='white', align='left')
    ax[1].set_title('Phân Bố Bậc Ra (Out-Degree) Tại Các Giao Lộ')
    ax[1].set_xlabel('Bậc ra (nhánh rẽ khả dĩ)')
    ax[1].set_ylabel('Số lượng nút')
    plt.tight_layout()
    plt.show()
else:
    print(f"Chưa tìm thấy {graph_path}. Vui lòng chạy build_graph trước.")

## 3. Phân Bố Tập Benchmark 30 Instances
Hiển thị tọa độ các điểm khách hàng của 30 instances phân tán trong bán kính 8km quanh Bếp Trung Tâm.

In [ ]:
from src.benchmark.instance_generator import load_benchmark_instances

instances = load_benchmark_instances('data/benchmark_instances')
print(f"Đã nạp {len(instances)} instances benchmark.")

if instances:
    plt.figure(figsize=(9, 9))
    depot = instances[0].stops[0]
    plt.plot(depot.lon, depot.lat, 'r*', markersize=18, label='Bếp Trung Tâm (Depot)', zorder=5)
    
    colors = plt.cm.tab20(np.linspace(0, 1, min(len(instances), 20)))
    for idx, inst in enumerate(instances[:10]):
        cust_lats = [s.lat for s in inst.stops[1:]]
        cust_lons = [s.lon for s in inst.stops[1:]]
        plt.scatter(cust_lons, cust_lats, color=colors[idx % len(colors)], alpha=0.6, s=35, label=f'{inst.name} ({len(cust_lats)} đơn)')
    
    plt.title('Bản Đồ Phân Bố Khách Hàng (10 Instances Tiêu Biểu TP.HCM)')
    plt.xlabel('Kinh độ (Longitude)')
    plt.ylabel('Vĩ độ (Latitude)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

## 4. Quá Trình Huấn Luyện RL (DQN & Double DQN Loss & Reward Curves)

In [ ]:
curves_path = 'rel_result/training_curves.csv'
if os.path.exists(curves_path):
    df_curves = pd.read_csv(curves_path)
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    
    for algo, group in df_curves.groupby('algorithm'):
        ax[0].plot(group['episode'], group['reward'], label=algo.upper(), marker='o', markersize=4)
        ax[1].plot(group['episode'], group['improvement_km'], label=algo.upper(), marker='s', markersize=4)
        
    ax[0].set_title('Đường Cong Phần Thưởng (Episode Reward)')
    ax[0].set_xlabel('Episode')
    ax[0].set_ylabel('Tổng Reward')
    ax[0].legend()
    
    ax[1].set_title('Quãng Đường Tối Ưu Được (km Saved)')
    ax[1].set_xlabel('Episode')
    ax[1].set_ylabel('km Tiết Kiệm Được')
    ax[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("Chưa có training_curves.csv. Sẽ hiển thị sau khi chạy train_cvrp_rl.")

## 5. Kết Quả Benchmark Giữa Các Chiến Lược (Summary Table & Boxplot)

In [ ]:
summary_path = 'rel_result/summary_by_algorithm.csv'
raw_path = 'rel_result/raw_runs.csv'

if os.path.exists(summary_path):
    df_sum = pd.read_csv(summary_path)
    display(df_sum[['protocol', 'strategy', 'mean_cost_km', 'std_cost_km', 'mean_runtime_ms', 'gap_to_best_pct', 'feasible_pct']])

if os.path.exists(raw_path):
    df_raw = pd.read_csv(raw_path)
    df_pA = df_raw[df_raw['protocol'] == 'A_realtime']
    
    plt.figure(figsize=(10, 6))
    strategies = df_pA['strategy'].unique()
    data = [df_pA[df_pA['strategy'] == s]['cost_km'].values for s in strategies]
    
    bplot = plt.boxplot(data, tick_labels=strategies, patch_artist=True, medianprops=dict(color='black', linewidth=1.5))
    colors = ['#90caf9', '#ef9a9a', '#a5d6a7', '#ffe082', '#b39ddb', '#80deea']
    for patch, col in zip(bplot['boxes'], colors):
        patch.set_facecolor(col)
        
    plt.title('So Sánh Tổng Chi Phí Tuyến (km) — Giao Thức A (Realtime 100 bước)')
    plt.ylabel('Tổng chiều dài tuyến (km)')
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

## 6. Kiểm Định Thống Kê Wilcoxon & Hiệu Chỉnh Holm-Bonferroni

In [ ]:
stat_path = 'rel_result/statistical_tests.csv'
if os.path.exists(stat_path):
    df_stat = pd.read_csv(stat_path)
    display(df_stat[['protocol', 'strategy', 'baseline', 'pct_improvement', 'wilcoxon_stat', 'raw_p_value', 'adj_p_holm', 'significant_05']])
else:
    print("Chưa có statistical_tests.csv. Vui lòng chạy analysis.py sau khi hoàn tất benchmark.")